In [16]:
from modules.ImageVectorDB import ImageVectorDB
import pandas as pd
import numpy as np

from warnings import filterwarnings

filterwarnings("ignore")

In [17]:
# Это для тестирования
db = ImageVectorDB(collection_name="wines_siglip2_b16_384")
points = db.get_all_vectors_and_metadata()
df = pd.DataFrame(points)
# df.to_csv('./from_qdrant.csv', index=False)
df.head(3)

INFO:httpx:HTTP Request: GET http://127.0.0.1:6333/collections "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://127.0.0.1:6333 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:6333/collections/wines_siglip2_b16_384/points/scroll "HTTP/1.1 200 OK"


Коллекция 'wines_siglip2_b16_384' уже существует.


INFO:httpx:HTTP Request: POST http://127.0.0.1:6333/collections/wines_siglip2_b16_384/points/scroll "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:6333/collections/wines_siglip2_b16_384/points/scroll "HTTP/1.1 200 OK"


,id,slug,vector,image_url
0,001b22f6-dad2-5a90-958b-f818d46c4e05,vinogradniki-gay-kodzora-terroir-de-gai-kodzor...,"[-0.049369235, -0.011733027, -0.023589728, -0....",https://api.vino-svoe.ru/v1/img/str-api/1160/1...
1,0030f5dd-f1b4-5958-9ccd-e6b7be3f4c6d,cabernet-franc-2024-one-barrel-by-dmitry-maslo...,"[-0.042438548, 0.026327018, -0.017997978, -0.0...",https://api.vino-svoe.ru/v1/img/str-api/1160/1...
2,00797fb6-f07d-59d9-a7b9-8df823277def,aromatnoe-shenen-blan-beloe-suhoe-13,"[-0.03801069, -0.0099736815, -0.02380107, -0.0...",https://api.vino-svoe.ru/v1/img/str-api/1160/1...


In [18]:
from modules.Searcher import search_similar

In [4]:
img_path = './dataset/test/images/37c03fb1-dd05-4670-b9a5-f106f71d49e4_jpg.rf.KDcs2pnhCSPaHcwYh0wi.jpg'
search_similar(img_path)

INFO:httpx:HTTP Request: POST http://127.0.0.1:6333/collections/wines_siglip2_b16_384/points/query "HTTP/1.1 200 OK"


[{'slug': 'nikolaev-i-synovya-festeliya-pti-verdo-krasnoe-suhoe-135',
  'score': 0.78734857},
 {'slug': 'le-k2-vassal-risling-reynskiy-beloe-suhoe-14', 'score': 0.78111017},
 {'slug': 'usadba-markoth-shardone-beloe-suhoe-12', 'score': 0.7730721},
 {'slug': 'tempelhof-winery-risling-risling-reynskiy-beloe-suhoe-115',
  'score': 0.7699772},
 {'slug': 'belmas-winery-risling-beloe-suhoe-135', 'score': 0.7679204}]

### Тут просто тесты с аугментациями стоковых фоток

In [5]:
import io
import random
import requests
from PIL import Image, ImageFilter, ImageEnhance
from torchvision import transforms

def download_and_augment(image_url: str) -> Image.Image:
    """
    Скачивает изображение по URL и применяет случайные аугментации:
    - небольшой блюр (GaussianBlur)
    - изменение яркости
    - растяжение (случайный resize с искажением пропорций)
    """
    # 1. Скачивание
    headers = {"User-Agent": "Mozilla/5.0 (compatible; WineIndexer/1.0)"}
    response = requests.get(image_url, headers=headers, timeout=30)
    response.raise_for_status()
    
    img = Image.open(io.BytesIO(response.content)).convert("RGB")
    
    # 2. Случайные трансформации
    
    # Блюр (с вероятностью 50%)
    if random.random() > 0.5:
        radius = random.uniform(0.5, 2.0)  # небольшой радиус
        img = img.filter(ImageFilter.GaussianBlur(radius=radius))
    
    # Яркость (с вероятностью 50%)
    if random.random() > 0.5:
        factor = random.uniform(0.7, 1.3)  # ±30% яркости
        enhancer = ImageEnhance.Brightness(img)
        img = enhancer.enhance(factor)
    
    return img

# Пример использования:
# img = download_and_augment("https://api.vino-svoe.ru/v1/img/str-api/1160/1...")
# vector = vectorizer.vectorize_images([img])[0]

In [6]:
img = download_and_augment(df.loc[0]['image_url'])
buffer = io.BytesIO()
img.save(buffer, format="PNG")
buffer

In [7]:
from modules.BottleCropper import BottleCropper
cropper = BottleCropper(model_path='models/yolo26s.pt', bottle_class_id=39, conf_thresh=0.35)
cropper.crop_image(buffer)

INFO:modules.BottleCropper:Загружаем модель: models/yolo26s.pt
INFO:modules.BottleCropper:Бутылка не найдена на изображении


In [9]:
results = []

for i, row in df.sample(frac=0.1, random_state=42).iterrows():
    img = download_and_augment(row['image_url'])
    true_slug = row['slug']
    
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")

    res = search_similar(buffer)

    results.append({
        "slug": true_slug,
        "found": res
    })

    break


INFO:httpx:HTTP Request: POST http://127.0.0.1:6333/collections/wines_siglip2_b16_384/points/query "HTTP/1.1 200 OK"


In [11]:
results

[{'slug': 'skalistyy-bereg-skalistyy-bereg-risling-beloe-suhoe-113',
  'found': [{'slug': 'skalistyy-bereg-skalistyy-bereg-risling-beloe-suhoe-113',
    'score': 0.99173856},
   {'slug': 'skalistyy-bereg-skalistyy-bereg-sovinon-blan-beloe-suhoe-12',
    'score': 0.90833974},
   {'slug': 'skalistyy-bereg-skalistyy-bereg-shardone-beloe-suhoe-135',
    'score': 0.89741695},
   {'slug': 'skalistyy-bereg-skalistyy-bereg-krasnaya-kniga-glava-2-merlo-krasnoe-suhoe-15',
    'score': 0.8874191},
   {'slug': 'skalistyy-bereg-skalistyy-bereg-pino-nuar-krasnoe-suhoe-13',
    'score': 0.8870322}]}]

In [12]:
rows = []
for item in results:
    query_slug = item['slug']
    for found in item['found']:
        rows.append({
            'query_slug': query_slug,
            'found_slug': found['slug'],
            'score': found['score']
        })

# Создаем DataFrame
df = pd.DataFrame(rows)

In [13]:
df

,query_slug,found_slug,score
0,skalistyy-bereg-skalistyy-bereg-risling-beloe-...,skalistyy-bereg-skalistyy-bereg-risling-beloe-...,0.991739
1,skalistyy-bereg-skalistyy-bereg-risling-beloe-...,skalistyy-bereg-skalistyy-bereg-sovinon-blan-b...,0.908340
2,skalistyy-bereg-skalistyy-bereg-risling-beloe-...,skalistyy-bereg-skalistyy-bereg-shardone-beloe...,0.897417
3,skalistyy-bereg-skalistyy-bereg-risling-beloe-...,skalistyy-bereg-skalistyy-bereg-krasnaya-kniga...,0.887419
4,skalistyy-bereg-skalistyy-bereg-risling-beloe-...,skalistyy-bereg-skalistyy-bereg-pino-nuar-kras...,0.887032


In [14]:
def evaluate_from_df(df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    # 1. Сортируем по запросу и по убыванию score (на случай, если порядок сбит)
    df = df.sort_values(['query_slug', 'score'], ascending=[True, False]).reset_index(drop=True)
    
    # 2. Добавляем ранг (позицию) внутри каждого запроса (начиная с 1)
    df['rank'] = df.groupby('query_slug').cumcount() + 1
    
    # 3. Помечаем, является ли найденный результат правильным (совпадение slug)
    df['is_correct'] = (df['query_slug'] == df['found_slug']).astype(int)
    
    # 4. Ограничиваем рассмотрение только топ-k результатами
    df_top_k = df[df['rank'] <= k].copy()
    
    # 5. Функция для расчёта метрик по одному запросу
    def calc_query_metrics(group):
        # Берем первую (самую высокую по score) правильную строку, если она есть
        correct = group[group['is_correct'] == 1]
        
        if len(correct) == 0:
            return pd.Series({'hit_rate': 0.0, 'mrr': 0.0, 'ndcg': 0.0})
        
        rank = correct['rank'].iloc[0]
        
        hit_rate = 1.0
        mrr = 1.0 / rank
        ndcg = 1.0 / np.log2(rank + 1)  # Для бинарной релевантности IDCG всегда = 1
        
        return pd.Series({'hit_rate': hit_rate, 'mrr': mrr, 'ndcg': ndcg})
    
    # 6. Применяем расчёт к каждому запросу
    metrics_per_query = df_top_k.groupby('query_slug').apply(calc_query_metrics).reset_index()
    
    # 7. Итоговые средние метрики
    final_scores = {
        'Hit-Rate@k': metrics_per_query['hit_rate'].mean(),
        'MRR': metrics_per_query['mrr'].mean(),
        f'NDCG@{k}': metrics_per_query['ndcg'].mean(),
        'Total_Queries': len(metrics_per_query)
    }
    
    print("--- Итоговые метрики ---")
    for name, value in final_scores.items():
        if name != 'Total_Queries':
            print(f"{name}: {value:.4f}")
        else:
            print(f"{name}: {value}")
            
    return metrics_per_query

In [15]:
evaluate_from_df(df, k=5)

NameError: name 'np' is not defined

In [153]:
len(df['query_slug'].unique())

211

In [12]:
import requests

r = requests.get(points[0]['image_url'])

In [17]:
import base64
from PIL import Image

Image.frombytes(base64.decode(r.content))

TypeError: decode() missing 1 required positional argument: 'output'

In [15]:
r

<Response [200]>

In [6]:
# import pandas as pd
import numpy as np

def calc_hit_rate(relevant: str, retrieved: list) -> int:
    return 1 if relevant in retrieved else 0

def calc_mrr(relevant: str, retrieved: list) -> float:
    for i, slug in enumerate(retrieved):
        if slug == relevant:
            return 1.0 / (i + 1)
    return 0.0

def calc_ndcg(relevant: str, retrieved: list, k: int = 10) -> float:
    relevances = [1.0 if slug == relevant else 0.0 for slug in retrieved[:k]]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(sorted(relevances, reverse=True)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_search(df: pd.DataFrame, db, k: int = 10, test_frac: float = 0.3):
    test_df = df.sample(frac=test_frac, random_state=42).reset_index(drop=True)
    
    metrics = {'hit_rate': [], 'mrr': [], 'ndcg': []}
    
    for _, row in test_df.iterrows():
        query_vector = row['vector']
        true_slug = row['slug']
        
        results = db.search_similar(query_vector=query_vector, limit=k)
        
        # Исключаем сам запрос из выдачи по slug, чтобы оценить поиск аналогов/дубликатов, а не тривиальное совпадение
        retrieved_slugs = [res['slug'] for res in results if res['slug'] != true_slug]
        
        metrics['hit_rate'].append(calc_hit_rate(true_slug, retrieved_slugs))
        metrics['mrr'].append(calc_mrr(true_slug, retrieved_slugs))
        metrics['ndcg'].append(calc_ndcg(true_slug, retrieved_slugs, k))

    print(f"--- Оценка качества поиска (Top-{k}, тест: {len(test_df)} записей) ---")
    print(f"Hit-Rate@{k}: {np.mean(metrics['hit_rate']):.4f}")
    print(f"MRR:          {np.mean(metrics['mrr']):.4f}")
    print(f"NDCG@{k}:     {np.mean(metrics['ndcg']):.4f}")
    
    return pd.DataFrame(metrics)

# Пример запуска:
# metrics_df = evaluate_search(df, db, k=10, test_frac=0.3)

In [7]:
results_df = evaluate_search(df, db, k=10, test_frac=0.3)

--- Оценка качества поиска (Top-10, тест: 633 записей) ---
Hit-Rate@10: 0.0000
MRR:          0.0000
NDCG@10:     0.0000


In [8]:
results_df

,hit_rate,mrr,ndcg
0,0,0.0,0.0
1,0,0.0,0.0
2,0,0.0,0.0
3,0,0.0,0.0
4,0,0.0,0.0
...,...,...,...
628,0,0.0,0.0
629,0,0.0,0.0
630,0,0.0,0.0
631,0,0.0,0.0
